In [2]:
### import all libraries

In [4]:
from langgraph.graph import START, END, StateGraph
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema, PydanticOutputParser
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_ollama import ChatOllama
from pydantic import BaseModel, Field
from typing import List, Annotated, TypedDict, Literal, List,Dict
from pyannote.audio import Pipeline
from pyannote.core import Segment
import librosa
import whisper
import os
import torch
import tempfile
import traceback

ModuleNotFoundError: No module named 'librosa'

In [ ]:
### define the state

In [ ]:
class MeetingAssistance(TypedDict):
    cleaned_audio: bytes
    converted_audio: str
    summary: str
    summary_evaluation: Literal["approved", "needs_improvement"]
    summary_feedback: str
    action_items: List[Dict[str, any]]
    action_items_evaluation: Literal["approved", "needs_improvement"]
    action_items_feedback: str
    key_decisions: List[Dict[str, any]]
    key_decisions_evaluation: Literal["approved", "needs_improvement"]
    key_decisions_feedback: str
    summary_iterations: int
    action_items_iterations: int
    key_decisions_iterations: int
    final_report: str
    max_iterations: int
    meeting_id: int

In [ ]:
class Evaluation(BaseModel):
    evaluation: Annotated[str, Literal["approved", "needs_improvement"]]
    feedback: Annotated[str, "The feedback of the LLM"]

In [ ]:
model = whisper.load_model("base")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token = os.environ["HUGGING_FACE_ACCESS_TOKEN"]
).to(device=device)

In [ ]:
generation_model = ChatOllama(model='qwen3:8b')
evaluation_model = ChatOllama(model='phi4-mini')

In [ ]:
action_items_schema = [
    ResponseSchema(name='speaker', description="The exact name of the speaker assigned to the action (e.g., SPEAKER_00). If it is a general team task, write 'Team'."),
    ResponseSchema(name='action_item', description="A clear, concise description of the task or promise made."),
    ResponseSchema(name='deadline', description="The deadline mentioned for the task. If no explicit deadline was mentioned, output 'Not Specified'."),
    ResponseSchema(name='status', description="The urgency of the action item. Classify strictly as 'High', 'Medium', or 'Low' based on the context.")
]

In [ ]:
key_decisions_schema = [
    ResponseSchema(name="topic", description="The main subject of the decision (e.g., Tech Stack, Budget). Keep it brief, like a category title."),
    ResponseSchema(name="decision", description="A clear description of what was finally agreed upon. Must be a finalized decision, not just a suggestion or debate."),
    ResponseSchema(name="speaker", description="The exact ID of the speaker who made the final call, proposed the accepted idea, or officially confirmed the decision (e.g., SPEAKER_02).")
]

In [ ]:
action_items_parser = StructuredOutputParser.from_response_schemas(response_schemas=action_items_schema)
key_decisions_parser = StructuredOutputParser.from_response_schemas(response_schemas=key_decisions_schema)
evaluation_parser = PydanticOutputParser(pydantic_object=Evaluation)

In [ ]:
def convert_audio(state: MeetingAssistance) -> MeetingAssistance:
    try:
        # 1. Grab the raw audio bytes from the State
        audio_bytes = state.get('cleaned_audio')
        
        if not audio_bytes:
            print("❌ Error: No audio bytes provided in state.")
            return {"converted_audio": "ERROR: No audio bytes found"}
        # 2. Write the bytes securely into a temporary file
        # We use a context manager to ensure it gets created properly
        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as temp_audio:
            temp_audio.write(audio_bytes)
            temp_path = temp_audio.name  # Get the path of the temporary file
        try:
            print(f"✅ Loaded raw bytes into temporary file: {temp_path}")
            
            # 3. Now we can proceed exactly like your old logic, but pointing to `temp_path`
            duration = librosa.get_duration(path=temp_path)
            print(f"⏱️ Audio duration: {duration:.2f} seconds")
            
            if duration < 10:
                print("⚠️ Audio too short for diarization, using transcription only.")
                result = model.transcribe(temp_path, fp16=False)
                segments = result["segments"]
                final_transcript = [f"UNKNOWN: {seg['text'].strip()}" for seg in segments]
                return {"converted_audio": "\n".join(final_transcript)}
            
            print("👥 Identifying speakers...")
            diarization = diarization_pipeline(temp_path)
            
            print(f"🎤 Whisper is transcribing...")
            result = model.transcribe(temp_path, fp16=False)
            segments = result["segments"]
            final_transcript = []
            
            for segment in segments:
                start_time = segment["start"]
                end_time = segment["end"]
                text = segment["text"].strip()
                
                try:
                    intersection = diarization.crop(Segment(start_time, end_time))
                    active_speakers = intersection.labels()
                    
                    if active_speakers:
                        speaker = active_speakers[0] 
                    else:
                        speaker = "UNKNOWN"
                except Exception as e:
                    print(f"Speaker detection error: {e}")
                    speaker = "UNKNOWN"
                    
                final_transcript.append(f"{speaker}: {text}")
            
            print("✅ Conversion Complete.")
            return {"converted_audio": "\n".join(final_transcript)}
        finally:
            # 4. CRITICAL: Clean up! Delete the temporary file so we don't leak storage 
            # This 'finally' block ensures deletion even if an error occurs above
            if os.path.exists(temp_path):
                os.remove(temp_path)
                print("🧹 Cleaned up temporary audio file.")
    
    except Exception as e:
        print(f"❌ Error in convert_audio: {str(e)}")
        traceback.print_exc()
        return {"converted_audio": f"ERROR: {str(e)}"}

In [ ]:
summary_prompt = ChatPromptTemplate.from_messages([
        ("system", """
    You are an advanced AI Meeting Assistant.
    Your objective is to read the provided diarized meeting transcript and generate a highly professional, accurate, and structured narrative summary of the meeting.

    You must strictly adhere to the following rules:
    1. **Accurate Attribution:** You MUST attribute specific ideas, proposals, agreements, and disagreements to the exact speaker identifiers provided in the script (e.g., SPEAKER_00, SPEAKER_01). Do not change, infer, or hallucinate their names.
    2. **Narrative Formatting:** Write the summary in complete, third-person narrative paragraphs. DO NOT output a dialogue script, chat log, or back-and-forth transcript format.
    3. **Exact Terminology:** You must retain the exact technical terms, acronyms, project names, and specific numbers used by the speakers in the meeting. Do not paraphrase critical metrics or technical jargon limits.
    4. **Zero Hallucination:** Base your summary strictly on the provided text. Do not add outside context, assume outcomes, or invent information. If a topic was brought up but left unresolved, state exactly that.
    5. **Professional Tone:** The output must be clear, objective, and logically grouped by the main topics discussed during the meeting.
    """),

    ("human", """
    Here is the diarized transcript for you to summarize:

    {converted_audio}

    Generate the detailed narrative summary below:
    """)
])


In [ ]:
action_items_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a meticulous AI Project Manager.
Your objective is to carefully analyze the provided diarized meeting transcript and extract all explicit action items, tasks, and promises made by the participants.
Strict Guidelines:
1. Only extract actual tasks or commitments. Do not extract general suggestions as action items.
2. For the 'speaker', use the exact speaker tags in the transcript (e.g., SPEAKER_01) who took ownership of the task.
3. Keep the 'action_item' description highly actionable (e.g., "Draft the Q3 marketing budget").
4. Do not hallucinate deadlines. If none were discussed, note it as "Not Specified".
5. Infer the 'status' (High, Medium, Low) based on the context of the conversation (e.g., "ASAP" or "Immediate" = High).
You must strictly output your response according to the JSON format instructions provided below. Do not include any extra conversational text outside of the JSON.
{format_instructions}
"""),
    ("human", """
Here is the diarized transcript to analyze:
{converted_audio}
""")
]).partial(format_instructions=action_items_parser.get_format_instructions())

In [ ]:
key_decisions_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a highly analytical AI Meeting Strategist.
Your objective is to read through the provided diarized meeting transcript and strictly extract the *finalized key decisions* and points of consensus reached by the team.
Strict Guidelines:
1. Only extract an item if there is a clear consensus, agreement, or a final call made. Do not extract ongoing debates, dropped ideas, or casual suggestions.
2. Group the outcome under a concise, professional 'topic' (e.g., 'Release Timeline').
3. For the 'speaker', identify the specific individual (e.g., SPEAKER_01) who proposed the winning idea or gave the final confirmation. 
4. Do not hallucinate decisions. If no concrete decisions were made in the meeting, return an empty array/list.
5. You must output the data exactly according to the JSON format instructions below. Do not include any conversational filler text.
{format_instructions}
"""),
    ("human", """
Here is the diarized transcript to analyze for key decisions:
{converted_audio}
""")
]).partial(format_instructions=key_decisions_parser.get_format_instructions())

In [ ]:
evaluation_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a ruthless, highly critical AI Quality Assurance Lead.
Your sole objective is to meticulously cross-reference an original diarized meeting transcript against a newly generated AI output (the {component_type}).
You must look for any excuse to fail the output. Approve it ONLY if it is completely flawless.
**Strict Rejection Criteria — Fail the output if:**
1. **Hallucination:** It contains numbers, names, deadlines, or topics that do not exist in the original transcript.
2. **Misattribution:** It assigns an idea, task, or decision to the wrong speaker.
3. **Critical Omission:** It completely ignores a major phase, argument, or final decision clearly present in the text.
4. **Poor Formatting:** It includes conversational filler text ("Here are the action items:"), conversational dialogue, or fails to be professional.
You must choose either `approved` (if absolutely flawless) or `needs_improvement` (if a single error exists).
If it `needs_improvement`, your feedback MUST be highly specific, directly pointing out exactly what was hallucinated, missed, or misattributed so that a secondary AI can correctly optimize it.
{format_instructions}
"""),
    ("human", """
**Original Diarized Transcript (The Ground Truth):**
{converted_audio}
=========================================
**Generated {component_type} to Evaluate:**
{generated_content}
""")
]).partial(format_instructions=evaluation_parser.get_format_instructions())

In [ ]:
summary_optimization_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are an advanced AI Meeting Assistant specializing in quality correction.
A previously generated meeting summary has failed Quality Assurance checks. Your objective is to read the original transcript, review the drafted summary, and carefully apply the QA Feedback to rewrite and fix the summary.
**You must strictly adhere to the following rules while rewriting:**
1. **Accurate Attribution:** You MUST attribute specific ideas, proposals, and disagreements to the exact speaker identifiers provided in the script (e.g., SPEAKER_00). Do not hallucinate their names.
2. **Narrative Formatting:** Write the summary in complete, third-person narrative paragraphs. DO NOT output a dialogue script or chat log format.
3. **Exact Terminology:** Retain the exact technical terms, acronyms, project names, and numbers used in the transcript.
4. **Zero Hallucination:** Fix the errors pointed out in the QA Feedback, but DO NOT add outside context or invent new information to do it. Base all corrections strictly on the transcript.
5. **Professional Tone:** The output must remain clear, objective, and logically grouped.
Do not include any conversational filler (e.g., "Here is the updated summary:"). Output ONLY the final, optimized summary paragraphs.
"""),
    ("human", """
**1. Original Diarized Transcript (Ground Truth):**
{converted_audio}
=========================================
**2. Drafted Summary (Failed QA):**
{current_summary}
=========================================
**3. QA Feedback (The issues you MUST fix):**
{feedback}
Please provide the fully corrected and optimized narrative summary below:
""")
])

In [ ]:
action_items_optimization_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a meticulous AI Project Manager specializing in quality correction.
A previously generated list of Action Items has failed Quality Assurance checks. Your objective is to read the original transcript, review the drafted action items, and carefully apply the QA Feedback to rewrite and fix the action items array.
**You must strictly adhere to the following rules while rewriting:**
1. Only extract actual tasks or commitments. Do not include general suggestions.
2. For the 'speaker', use the exact speaker tags in the transcript (e.g., SPEAKER_01) who took ownership of the task. Do not hallucinate names.
3. Keep the 'action_item' description highly actionable.
4. Do not hallucinate deadlines. If none were discussed, note it as "Not Specified".
5. Infer the 'status' (High, Medium, Low) based on the context of the conversation.
**Critical Instruction:** Fix the explicit errors pointed out in the QA Feedback based ONLY on the original transcript. 
You must strictly output your revised response according to the JSON format instructions provided below. Do not include any extra conversational text.
{format_instructions}
"""),
    ("human", """
**1. Original Diarized Transcript (Ground Truth):**
{converted_audio}
=========================================
**2. Drafted Action Items (Failed QA):**
{current_action_items}
=========================================
**3. QA Feedback (The issues you MUST fix):**
{feedback}
Please provide the fully corrected json output below:
""")
]).partial(format_instructions=action_items_parser.get_format_instructions())


In [ ]:
key_decisions_optimization_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a highly analytical AI Meeting Strategist specializing in quality correction.
A previously generated list of Key Decisions has failed Quality Assurance checks. Your objective is to read the original transcript, review the drafted key decisions, and carefully apply the QA Feedback to rewrite and fix the decisions array.
**You must strictly adhere to the following rules while rewriting:**
1. Only extract an item if there is a clear consensus, agreement, or a final call made. Do not extract ongoing debates or dropped ideas.
2. Group the outcome under a concise, professional 'topic' (e.g., 'Release Timeline').
3. For the 'speaker', identify the specific individual (e.g., SPEAKER_02) who proposed the winning idea or gave the final confirmation. 
4. Do not hallucinate decisions. Base all corrections explicitly on the transcript text.
**Critical Instruction:** Fix the exact errors pointed out in the QA Feedback without introducing any new facts. 
You must strictly output your revised response according to the JSON format instructions provided below. Do not include any conversational filler text.
{format_instructions}
"""),
    ("human", """
**1. Original Diarized Transcript (Ground Truth):**
{converted_audio}
=========================================
**2. Drafted Key Decisions (Failed QA):**
{current_key_decisions}
=========================================
**3. QA Feedback (The issues you MUST fix):**
{feedback}
Please provide the fully corrected json output below:
""")
]).partial(format_instructions=key_decisions_parser.get_format_instructions())


In [ ]:
def generate_summary(state: MeetingAssistance) -> MeetingAssistance:
    chain = summary_prompt | generation_model | StrOutputParser()
    summary = chain.invoke({"converted_audio": state['converted_audio']})
    return {"summary": summary}

In [ ]:
def generate_action_items(state: MeetingAssistance) -> MeetingAssistance:
    chain = action_items_prompt | generation_model | action_items_parser
    action_items = chain.invoke({'converted_audio': state['converted_audio']})
    return {"action_items": action_items}

In [ ]:
def generate_key_decisions(state: MeetingAssistance) -> MeetingAssistance:
    chain = key_decisions_prompt | generation_model | key_decisions_parser
    key_decisions = chain.invoke({"converted_audio": state['converted_audio']})
    return {"key_decisions": key_decisions}

In [ ]:
def evaluate_summary(state: MeetingAssistance) -> MeetingAssistance:
    chain = evaluation_prompt | evaluation_model | evaluation_parser
    result = chain.invoke({
        "component_type":"Summary",
        "converted_audio": state['converted_audio'], 
        "generated_content": state['summary']
    })
    return {"summary_evaluation": result['evaluation'], "summary_feedback": result['feedback']}

In [ ]:
def optimize_summary(state: MeetingAssistance) -> MeetingAssistance:
    chain = summary_optimization_prompt | generation_model | StrOutputParser()
    summary = chain.invoke({
        "converted_audio": state['converted_audio'],
        "current_summary": state['summary'],
        "feedback": state['summary_feedback']   
        })
    summary_iteration = state['summary_iterations'] + 1
    return {"summary": summary, "summary_iterations": summary_iteration}

In [ ]:
def evaluate_action_items(state: MeetingAssistance) -> MeetingAssistance:
    chain = evaluation_prompt | evaluation_model | evaluation_parser
    result = chain.invoke({
        "component_type": "Action Items",
        "converted_audio": state['converted_audio'],
        "generated_content": state['action_items']
    })
    return {"action_items_evaluation": result['evaluation'], "action_items_feedback": result['feedback']}

In [ ]:
def optimize_action_items(state: MeetingAssistance) -> MeetingAssistance:
    chain = action_items_optimization_prompt | generation_model | action_items_parser
    result = chain.invoke({
        "converted_audio": state['converted_audio'],
        "current_action_items": state['action_items'],
        "feedback": state['action_items_feedback']
    })
    action_items_iteration = state['action_items_iterations'] + 1
    return {"action_items": result, "action_items_iterations": action_items_iteration}

In [ ]:
def evaluate_key_decisions(state: MeetingAssistance) -> MeetingAssistance:
    chain = evaluation_prompt | evaluation_model | evaluation_parser
    result = chain.invoke({
        "component_type":"Key Decisions",
        "converted_audio": state['converted_audio'], 
        "generated_content": state['key_decisions']
    })
    return {"key_decisions_evaluation": result.evaluation, "key_decisions_feedback": result.feedback}

In [ ]:
def optimize_key_decisions(state: MeetingAssistance) -> MeetingAssistance:
    chain = key_decisions_optimization_prompt | generation_model | key_decisions_parser
    result = chain.invoke({
        "converted_audio": state['converted_audio'],
        "current_key_decisions": state['key_decisions'],
        "feedback": state['key_decisions_feedback']
    })
    key_decision_iteration = state['key_decisions_iterations'] + 1
    return {"key_decisions": result, "key_decisions_iterations": key_decision_iteration}

In [ ]:
def format_text(state: MeetingAssistance) -> MeetingAssistance:
    """
        This node creates the final string that the user actually sees.
    """
    # 1. Get data from State
    summary_text = state.get("summary", "No summary available.")
    actions_list = state.get("action_items", [])
    decisions_list = state.get("key_decisions", [])

    # 2. Format Action Items 
    formatted_actions = ""
    # If it's a string, just use it. Otherwise, loop through the List of Dictionaries.
    if isinstance(actions_list, str):
        formatted_actions = actions_list
    elif actions_list and len(actions_list) > 0:
        for item in actions_list:
            formatted_actions += f"- **Action Item:** {item.get('action_item', 'N/A')} | **Speaker:** {item.get('speaker', 'Unknown')} | **Deadline:** {item.get('deadline', 'N/A')} | **Status:** {item.get('status', 'N/A')}\n"
    else:
        formatted_actions = "No action items identified."

    # 3. Format Key Decisions 
    formatted_decisions = ""
    if isinstance(decisions_list, str):
        formatted_decisions = decisions_list
    elif decisions_list and len(decisions_list) > 0:
        for item in decisions_list:
            formatted_decisions += f"- **Topic:** {item.get('topic', 'N/A')} | **Decision:** {item.get('decision', 'N/A')} | **Speaker:** {item.get('speaker', 'Unknown')}\n"
    else:
        formatted_decisions = "No key decisions recorded."

    # 4. Combine everything into the User View
    user_view = f"""
=========================================
          MEETING ANALYSIS REPORT
=========================================

SUMMARY:
{summary_text}

-----------------------------------------
ACTION ITEMS:
{formatted_actions}

-----------------------------------------
KEY DECISIONS:
{formatted_decisions}
=========================================
"""
    
    # Save to the final state field (assuming you add 'final_report' to your TypedDict)
    return {"final_report": user_view}


In [ ]:
def check_condition(state: MeetingAssistance):
    if(state['summary_evaluation'] == "approved" or state['summary_iterations'] > state['max_iterations']):
        return "approved"
    
    elif(state['action_items_evaluation'] == "approved" or state['action_items_iterations'] > state['max_iterations']):
        return "approved"
    
    elif(state['key_decisions_evaluation'] == "approved" or state['key_decisions_iterations'] > state['max_iterations']):
        return "approved"
    
    return "needs_improvement"

In [ ]:
## define the graph

In [ ]:
from src.utils import check_summary, check_action_items, check_key_decisions

In [ ]:
graph = StateGraph(state_schema=MeetingAssistance)
## define the nodes
graph.add_node("convert_audio", convert_audio)
graph.add_node("generate_summary", generate_summary)
graph.add_node("generate_action_items", generate_action_items)
graph.add_node("generate_key_decisions", generate_key_decisions)
graph.add_node("evaluate_summary", evaluate_summary)
graph.add_node("optimize_summary", optimize_summary)
graph.add_node("evaluate_action_items", evaluate_action_items)
graph.add_node("optimize_action_items", optimize_action_items)
graph.add_node("evaluate_key_decisions", evaluate_key_decisions)
graph.add_node("optimize_key_decisions", optimize_key_decisions)
graph.add_node("format_text", format_text)
## define the edges
## define the edges
graph.add_edge(START, "convert_audio")

# Fan-out: Start all three generations in parallel after audio conversion
graph.add_edge("convert_audio", "generate_summary")
graph.add_edge("convert_audio", "generate_action_items")
graph.add_edge("convert_audio", "generate_key_decisions")

# Independent Refinement Loops for each component
graph.add_edge("generate_summary", "evaluate_summary")
graph.add_conditional_edges("evaluate_summary", check_summary, {"approved": "format_text", "needs_improvement": "optimize_summary"})
graph.add_edge("optimize_summary", "evaluate_summary")

graph.add_edge("generate_action_items", "evaluate_action_items")
graph.add_conditional_edges("evaluate_action_items", check_action_items, {"approved": "format_text", "needs_improvement": "optimize_action_items"})
graph.add_edge("optimize_action_items", "evaluate_action_items")

graph.add_edge("generate_key_decisions", "evaluate_key_decisions")
graph.add_conditional_edges("evaluate_key_decisions", check_key_decisions, {"approved": "format_text", "needs_improvement": "optimize_key_decisions"})
graph.add_edge("optimize_key_decisions", "evaluate_key_decisions")

# Convergence: All paths wait for each other at format_text
graph.add_edge("format_text", END)

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
MeetingAssistance

In [ ]:
# workflow.invoke({"cleaned_audio": 'C:\Meeting Assistance\notebook\ssgl-171-pandora.mp3'})